# Suite PRX — Proxies

The dynamic implementation: registry, read-only instances, the builder DSL, create/clone/update, relation entries,
schema inference through links, entry removal, and the behaviors FRAMEWORK.md pins down.

In [ ]:
import math

from mbse.Schemas.Framework import Plain, Proxies, Reachable, Schemas as S, Validators
from support import entries, get_property, raises, set_property, text

Person = S.OfObject.Builder().ref().properties(text("name"), text("age", int), text("score", float), text("ok", bool),
                                         text("blob", bytes)).create()
Pet = S.OfObject.Builder().ref().properties(text("name")).create()
Robot = S.OfObject.Builder().ref().properties(text("name")).create()
Owns = S.OfRelation.Builder().links("owner", "item").properties(text("since"), text("share", float),
                                                                   text("rank", int), text("flag", bool)).create()
Knows = S.OfRelation.Builder().links("a", "b").create()
Trio = S.OfRelation.Builder().links("x", "y", "z").create()
Person = S.OfObject.Builder(Person).relations(
    lambda r: r.name("pets").of(Owns).me("owner"),
    lambda r: r.name("knows").of(Knows).me("a"),
    lambda r: r.name("known_by").of(Knows).me("b"),
    lambda r: r.name("trio").of(Trio).me("x"),
).update()
Pet = S.OfObject.Builder(Pet).relations(lambda r: r.name("owners").of(Owns).me("item"),
                                        lambda r: r.name("trio_y").of(Trio).me("y")).update()
Robot = S.OfObject.Builder(Robot).relations(lambda r: r.name("owners").of(Owns).me("item"),
                                            lambda r: r.name("trio_z").of(Trio).me("z")).update()
for name, schema in [("Person", Person), ("Pet", Pet), ("Robot", Robot), ("Owns", Owns), ("Knows", Knows),
                     ("Trio", Trio)]:
    Proxies.register(name, schema)
B = Proxies.Builders

## PRX-01 · Registry

In [ ]:
with raises(ValueError, match="already registered"):
    Proxies.register("Person", Person)
with raises(AttributeError, match="no schema registered"):
    B.Nobody()
with raises(TypeError, match="no relation builder"):
    B.Owns()
assert Proxies.schema("Person") is Person and B.schema("Person") is Person
assert getattr(B, "then", None) is None  # not a registered name
assert Proxies.name_of(Owns) == "Owns" and B.name_of(Person) == "Person"
with raises(LookupError):
    Proxies.name_of(S.OfObject.Data())
with raises(TypeError):
    Proxies.schema("Owns")  # a relation is not an object schema

Proxies.register("iso3166.Country", S.OfObject.Builder().ref().properties(text("alpha2")).create())
fr = getattr(B, "iso3166.Country")().alpha2("FR").create()
assert fr.schema_name() == "iso3166.Country" and fr.alpha2 == "FR"

## PRX-02 · Instances: read-only attributes; unset properties raise `AttributeError`

In [ ]:
p = B.Person().name("Ada").create()
assert p.name == "Ada"
with raises(AttributeError, match="'age' is not set"):
    p.age
with raises(AttributeError):
    p.no_such_property
assert not hasattr(p, "age") and getattr(p, "age", "default") == "default"
with raises(AttributeError, match="read-only"):
    p.name = "Eve"
with raises(AttributeError, match="read-only"):
    p.age = 1  # also for properties that are not set
assert p.name == "Ada"
assert p.schema_name() == "Person" and p.identity() == id(p)
alive = [B.Person().create() for _ in range(100)]
assert len({o.identity() for o in alive}) == 100  # distinct among live objects
# identity() is id(self): an object that has been garbage-collected may share an identity with a later one.
# Serializers only compare identities of objects alive during one pass, so this is safe there.

## PRX-03 · Setters: a value, or a `Spec` given a `Visitors.OfNative`

In [ ]:
q = (B.Person()
     .name("Bo")
     .age(lambda v: v.set(41))
     .score(lambda v: v.set(1.5).clear().set(2.5))  # the Spec may chain
     .ok(False)
     .blob(b"")
     .create())
assert (q.name, q.age, q.score, q.ok, q.blob) == ("Bo", 41, 2.5, False, b"")
seen = []
B.Person(q).age(lambda v: seen.append((v.has(), v.get())))
assert seen == [(True, 41)]
B.Person(q).name(lambda v: v.clear()).update()
with raises(AttributeError):
    q.name
B.Person(q).name(lambda v: v.clear()).update()  # clearing an absent property is harmless
with raises(AttributeError, match="not a property or adjacency"):
    B.Person().nickname("x")
with raises(AttributeError):
    B.Person()._private("x")  # names starting with "_" are never DSL setters

## PRX-04 · Finding F2 (pinned): names that collide with builder or instance members

In [ ]:
Odd = S.OfObject.Builder().ref().properties(text("create"), text("accept"), text("_values"), text("$ref")).create()
Proxies.register("Odd", Odd)
with raises(TypeError):
    B.Odd().create("x")  # the builder method wins; the DSL cannot set 'create'
b = B.Odd()
for name, value in [("create", "C"), ("accept", "A"), ("_values", "V"), ("$ref", "R")]:
    set_property(b, name, value)  # the visitor protocol reaches every name
odd = b.create()
assert getattr(odd, "$ref") == "R"
assert callable(odd.accept)  # the instance method wins over the property
assert odd._values != "V"  # an internal attribute shadows the property (a bug-shaped wart; see FRAMEWORK.md)
assert Plain.ToPlain(Odd, odd)["objects"]["s0"] == {"create": "C", "accept": "A", "_values": "V", "$ref": "R"}

## PRX-05 · Finalization: create / clone / update

In [ ]:
src = B.Person().name("Src").age(1).create()
with raises(ValueError):
    B.Person(src).create()
with raises(ValueError):
    B.Person().clone()
with raises(ValueError):
    B.Person().update()
with raises(TypeError, match="not a 'Person'"):
    B.Person(B.Pet().name("x").create())
c = B.Person(src).age(2).clone()
assert c is not src and (c.name, c.age, src.age) == ("Src", 2, 1)
u = B.Person(src).age(3).update()
assert u is src and src.age == 3
reused = B.Person().name("Twin")
t1, t2 = reused.create(), reused.create()
assert t1 is not t2 and t1.name == t2.name == "Twin"
pending = B.Person(src).age(99)
assert src.age == 3  # nothing changes until update()

## PRX-06 · Finding F8 (pinned): builders do not validate native types; serialization and validation do

In [ ]:
sloppy = B.Person().age("forty").ok(1).create()
assert sloppy.age == "forty" and sloppy.ok == 1
with raises(TypeError):
    Plain.ToPlain(Person, sloppy)
problems = Validators.Validate(B)(Person, sloppy)
assert "Person#0.age: expected int, got str" in problems and "Person#0.ok: expected bool, got int" in problems

## PRX-07 · Entries: added through adjacencies, visible from every end

In [ ]:
ann = B.Person().name("Ann").create()
rex = B.Pet().name("Rex").create()
B.Person(ann).pets(lambda x: x.item(rex).since("2020-01-01")).update()
assert [e["item"] for e in entries(Person, ann, "pets")] == [rex]
assert [(e["owner"], e["since"]) for e in entries(Pet, rex, "owners")] == [(ann, "2020-01-01")]
with raises(TypeError, match="entry Spec"):
    B.Person().pets(rex)
with raises(TypeError):
    B.Person().pets(item=rex)
with raises(AttributeError, match="not a link this entry can set"):
    B.Person().pets(lambda x: x.owner(ann))  # the object's own link is implied
with raises(AttributeError):
    B.Person().pets(lambda x: x.item(rex).colour("red"))
with raises(ValueError, match="link 'item' is not set"):
    B.Person().name("Nobody").pets(lambda x: x.since("never")).create()
with raises(TypeError, match="only link proxy instances"):
    B.Person().pets(lambda x: x.item("Rex"))

## PRX-08 · Entries form a set: equality follows schema equality

In [ ]:
def count(obj, adjacency, schema=Person):
    return len(entries(schema, obj, adjacency))


o = B.Person().name("Set").create()
add = lambda **props: B.Person(o).pets(lambda x: [x.item(rex)] + [getattr(x, k)(v) for k, v in props.items()] and x).update()
add()
add()
assert count(o, "pets") == 1  # an identical entry is elided
add(since="a")
add(since="a")
add(since="b")
assert count(o, "pets") == 3  # absent, "a", "b"
add(share=0.0)
add(share=-0.0)
add(share=math.nan)
add(share=float("nan"))
assert count(o, "pets") == 6  # 0.0 and -0.0 differ; all NaNs are equal
add(rank=1)
add(flag=True)
assert count(o, "pets") == 8  # int 1 and bool True are different entries

## PRX-09 · Inline creation through a link needs an unambiguous schema

In [ ]:
Lonely = S.OfRelation.Builder().links("p", "orphan").create()
Proxies.register("Lonely", Lonely)
Person2 = S.OfObject.Builder().ref().properties(text("name")).relations(lambda r: r.name("lonely").of(Lonely).me("p")).create()
Proxies.register("Person2", Person2)

with raises(TypeError, match="exactly one object schema"):
    B.Person().pets(lambda x: x.item(lambda y: y.name("?")))  # Pet and Robot both fill 'item'
with raises(TypeError, match="found []"):
    B.Person2().lonely(lambda x: x.orphan(lambda y: y))  # nothing fills 'orphan'
kid = B.Pet().name("Kid").create()
owner = B.Person().name("Own").pets(lambda x: x.item(kid)).trio(lambda x: x.y(lambda y: y.name("Yy")).z(lambda z: z.name("Zz"))).create()
trio = entries(Person, owner, "trio")[0]
assert trio["y"].schema_name() == "Pet" and trio["z"].schema_name() == "Robot"

## PRX-10 · Nested inline objects are created each time the outer builder is finalized

In [ ]:
outer = B.Person().name("Outer").trio(lambda x: x.y(lambda y: y.name("Y")).z(lambda z: z.name("Z")))
first, second = outer.create(), outer.create()
assert entries(Person, first, "trio")[0]["y"] is not entries(Person, second, "trio")[0]["y"]

## PRX-11 · Removing entries, and update() writing entries back exactly

In [ ]:
r = B.Person().name("Rem").pets(lambda x: x.item(rex).since("1")).pets(lambda x: x.item(rex).since("2")).knows(lambda x: x.b(ann)).create()


def since_of(entry):
    return get_property(entry, "since") if entry.has("since") else None


B.Person(r).adjacency("pets", lambda a: a.entries(lambda e: a.remove(e) if since_of(e) == "1" else None)).update()
assert [e["since"] for e in entries(Person, r, "pets")] == ["2"]
assert [e["b"] for e in entries(Person, r, "knows")] == [ann]  # other adjacencies are untouched
assert all(e["owner"] is not r or e["since"] == "2" for e in entries(Pet, rex, "owners"))  # gone from the other end
with raises(AttributeError, match="not an adjacency"):
    B.Person(r).adjacency("nope", lambda a: None)

## PRX-12 · Finding F4 (pinned): concurrent builders — the last update() wins, entries included

In [ ]:
lw = B.Person().name("LW").create()
b1 = B.Person(lw).pets(lambda x: x.item(rex).rank(1))
b2 = B.Person(lw).pets(lambda x: x.item(rex).rank(2))
b1.update()
b2.update()
assert [e["rank"] for e in entries(Person, lw, "pets")] == [2]

## PRX-13 · clone() copies entries; Finding F5 (pinned): a self-loop's clone links back to the original

In [ ]:
orig = B.Person().name("Orig").pets(lambda x: x.item(rex)).knows(lambda x: x.b(ann)).create()
cl = B.Person(orig).name("Clone").clone()
assert [e["item"] for e in entries(Person, cl, "pets")] == [rex]
assert {e["owner"].name for e in entries(Pet, rex, "owners")} >= {"Orig", "Clone"}

loop = B.Person().name("Loop").create()
B.Person(loop).knows(lambda x: x.b(loop)).update()
assert [e["b"] for e in entries(Person, loop, "knows")] == [loop]
assert [e["a"] for e in entries(Person, loop, "known_by")] == [loop]
loop2 = B.Person(loop).name("Loop2").clone()
assert [e["b"] for e in entries(Person, loop2, "knows")] == [loop]
assert [e["a"] for e in entries(Person, loop2, "known_by")] == [loop]

## PRX-14 · Self-relations: one entry, seen through both adjacencies of the same schema

In [ ]:
x = B.Person().name("X").create()
y = B.Person().name("Y").knows(lambda e: e.b(x)).create()
assert [e["b"] for e in entries(Person, y, "knows")] == [x]
assert [e["a"] for e in entries(Person, x, "known_by")] == [y]
assert entries(Person, x, "knows") == [] and entries(Person, y, "known_by") == []

## PRX-15 · The builder's visitor API reads and edits pending state

In [ ]:
vb = B.Person().name("V").age(5)
names = []
vb.properties(lambda prop: names.append((prop.name(), prop.has())))
assert names == [("name", True), ("age", True)]
assert vb.has("name") and not vb.has("score")
assert get_property(vb, "age") == 5
vb.clear("age")
assert not vb.has("age")
with raises(AttributeError, match="not a property"):
    vb.property("nope", lambda prop: None)
adjacencies = []
vb.adjacencies(lambda a: adjacencies.append((a.name(), a.me())))
assert adjacencies == [("pets", "owner"), ("knows", "a"), ("known_by", "b"), ("trio", "x")]

## PRX-16 · `Proxies.OfObject.Builder` builds directly; `Proxies.OfObject.Data` is the instance type

In [ ]:
direct = Proxies.OfObject.Builder(Person, "Person").name("Direct").create()
assert isinstance(direct, Proxies.OfObject.Data) and direct.name == "Direct"
assert not isinstance(Proxies.OfObject.Builder(Person, "Person"), Proxies.OfObject.Data)
assert Proxies.OfObject.Builder(Person, "Person", direct).name("Again").update() is direct and direct.name == "Again"
from support import Fake

with raises(TypeError, match="a builder's source must be a proxy instance"):
    B.Person(Fake("Person", {"name": "fake"}))

## PRX-17 · Entry properties must be native values (entries are keyed by them)
Object properties accept anything until serialized (F8), but an entry's values form its identity in the relation's
table, so a non-native value is refused when the entry is stored.

In [ ]:
for bad, name in [(["a"], "list"), ({"k": 1}, "dict"), (None, "NoneType"), (bytearray(b"x"), "bytearray")]:
    with raises(TypeError, match=f"an entry property must be a native value or a value object, got {name}"):
        B.Person().pets(lambda x, bad=bad: x.item(rex).since(bad)).create()
# Bytes are native, so they key entries like any other value: equal bytes are one entry. (Keys are not type-checked
# against the schema until serialized, so the entries are counted on the builder rather than through Plain.)
k = B.Person().pets(lambda x: x.item(rex).since(b"a")).pets(lambda x: x.item(rex).since(b"a")).pets(lambda x: x.item(rex).since(b"b")).create()
found = []
B.Person(k).adjacency("pets", lambda a: a.entries(found.append))
assert len(found) == 2